In [3]:
import os
print(os.environ.get("SSL_CERT_FILE"))

C:\Users\ashutoshawasthi\Desktop\FineTunning\ft\Library\ssl\cacert.pem


In [4]:
import os
os.environ.pop("SSL_CERT_FILE", None)

'C:\\Users\\ashutoshawasthi\\Desktop\\FineTunning\\ft\\Library\\ssl\\cacert.pem'

In [5]:
import os
import certifi

os.environ["SSL_CERT_FILE"] = certifi.where()

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer
from transformers import AutoModelForCausal

c:\Users\ashutoshawasthi\Desktop\FineTunning\ft\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [7]:
dataset = load_dataset("json",data_files="dataset.json")
dataset

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 5
    })
})

In [8]:
print(dataset["train"][0])

{'instruction': 'Translate English to French', 'input': 'Hello', 'output': 'Bonjour'}


In [9]:
def format_prompt(example):
    prompt = f"""
###Instruction:
{example["instruction"]}
###Input:
{example["input"]}
###Response:
{example["output"]}
"""
    return {"text" : prompt}

In [10]:
dataset  = dataset.map(format_prompt)

In [11]:
dataset

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output', 'text'],
        num_rows: 5
    })
})

In [12]:
dataset["train"][0]

{'instruction': 'Translate English to French',
 'input': 'Hello',
 'output': 'Bonjour',
 'text': '\n###Instruction:\nTranslate English to French\n###Input:\nHello\n###Response:\nBonjour\n'}

In [13]:
tokenizer = AutoTokenizer.from_pretrained("distilgpt2")


In [14]:
tokenizer.pad_token = tokenizer.eos_token

In [15]:
def tokenize_function(example):
    tokens = tokenizer(
        example["text"],
        truncation= True,
        padding = "max_length",
        max_length = 128 
    )
    return tokens

In [16]:
tokenized_dataset = dataset.map(tokenize_function)

Map: 100%|██████████| 5/5 [00:00<00:00, 125.03 examples/s]


In [22]:
tokenized_dataset["train"]

Dataset({
    features: ['instruction', 'input', 'output', 'text', 'input_ids', 'attention_mask'],
    num_rows: 5
})